# Week 1: OpenCV Preprocessing Pipeline for Prescription Parsing

Medical doctor notes present unique computer vision challenges:
1. **Mobile Photos vs Scans**: Lighting gradients, finger/phone shadows, lens vignetting, skew.
2. **Degraded Handwriting**: Varying pen pressure, broken ink strokes, paper grain.
3. **Fluctuating Text Lines**: Unlike printed text with rigid grids, handwriting meanders.

In this notebook, we walk through our modular Stage 1 Preprocessing pipeline:
- **Illumination & Shadow Removal**: Morphological background division + CLAHE
- **Deskewing**: Horizontal projection profile variance & affine rotation
- **Binarization**: Bilateral denoising + Sauvola thresholding + connected component speckle cleaning
- **Segmentation**: Morphological horizontal smearing for line and word crops

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import cv2
import numpy as np
import matplotlib.pyplot as plt

from src.preprocessing.illumination import correct_illumination, remove_shadows_morphological
from src.preprocessing.deskew import deskew, rotate_image
from src.preprocessing.binarization import sauvola_threshold, binarize, clean_speckles, denoise_bilateral
from src.preprocessing.segmentation import segment_prescription
from src.preprocessing.pipeline import PreprocessingPipeline
from src.utils.visualizer import plot_preprocessing_stages

%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

## 1. Load Sample Images (Mobile Photo vs Scan)

In [ ]:
photo_path = '../data/raw/prescription_photo.png'
scan_path = '../data/raw/prescription_scan.png'

img_photo = cv2.imread(photo_path)
img_scan = cv2.imread(scan_path)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
ax1.imshow(cv2.cvtColor(img_photo, cv2.COLOR_BGR2RGB))
ax1.set_title("Raw Phone Photo (Skew + Shadows)", fontsize=12)
ax1.axis('off')

ax2.imshow(cv2.cvtColor(img_scan, cv2.COLOR_BGR2RGB))
ax2.set_title("Clean Scan", fontsize=12)
ax2.axis('off')
plt.show()

## 2. Illumination & Shadow Normalization
Using morphological background division to estimate uneven paper illumination and divide foreground ink by background paper.

In [ ]:
photo_corrected = correct_illumination(img_photo, use_morphology=True, use_clahe=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
ax1.imshow(cv2.cvtColor(img_photo, cv2.COLOR_BGR2RGB))
ax1.set_title("Before Shadow Removal", fontsize=12)
ax1.axis('off')

ax2.imshow(cv2.cvtColor(photo_corrected, cv2.COLOR_BGR2RGB))
ax2.set_title("After Morphological Division + CLAHE", fontsize=12)
ax2.axis('off')
plt.show()

## 3. Deskewing via Projection Profile Variance

In [ ]:
deskewed_img, angle = deskew(photo_corrected, method='projection')
print(f"Detected Skew Angle: {angle:+.2f} degrees")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 8))
ax1.imshow(cv2.cvtColor(photo_corrected, cv2.COLOR_BGR2RGB))
ax1.set_title("Tilted Input", fontsize=12)
ax1.axis('off')

ax2.imshow(cv2.cvtColor(deskewed_img, cv2.COLOR_BGR2RGB))
ax2.set_title(f"Deskewed ({angle:+.2f}° rotation corrected)", fontsize=12)
ax2.axis('off')
plt.show()

## 4. Denoising & Binarization (Sauvola vs Otsu vs Adaptive Gaussian)

In [ ]:
gray = cv2.cvtColor(deskewed_img, cv2.COLOR_BGR2GRAY)
denoised = denoise_bilateral(gray, d=7, sigma_color=50, sigma_space=50)

bin_otsu = binarize(denoised, method='otsu')
bin_adapt = binarize(denoised, method='adaptive_gaussian', block_size=25, c_param=10)
bin_sauvola = binarize(denoised, method='sauvola', block_size=25, sauvola_k=0.2)
bin_clean = clean_speckles(bin_sauvola, min_area=8)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(bin_otsu, cmap='gray')
axes[0, 0].set_title("Otsu (Global)", fontsize=12)
axes[0, 0].axis('off')

axes[0, 1].imshow(bin_adapt, cmap='gray')
axes[0, 1].set_title("Adaptive Gaussian", fontsize=12)
axes[0, 1].axis('off')

axes[1, 0].imshow(bin_sauvola, cmap='gray')
axes[1, 0].set_title("Sauvola (Gold standard for handwriting)", fontsize=12)
axes[1, 0].axis('off')

axes[1, 1].imshow(bin_clean, cmap='gray')
axes[1, 1].set_title("Sauvola + Speckle Cleaning", fontsize=12)
axes[1, 1].axis('off')
plt.tight_layout()
plt.show()

## 5. Line & Word Segmentation
Segmenting lines and words prepares isolated image patches for Tesseract (baseline) and TrOCR (transformer) in Weeks 1-2.

In [ ]:
lines = segment_prescription(bin_clean, deskewed_img, method='morphological', segment_words=True)
print(f"Extracted {len(lines)} line strips.")

# Display first 5 extracted line crops
fig, axes = plt.subplots(min(5, len(lines)), 1, figsize=(10, 8))
for i in range(min(5, len(lines))):
    line = lines[i]
    axes[i].imshow(cv2.cvtColor(line.image, cv2.COLOR_BGR2RGB))
    axes[i].set_title(f"Line {line.line_index + 1} ({len(line.words)} words detected)", fontsize=10)
    axes[i].axis('off')
plt.tight_layout()
plt.show()

## 6. End-to-End Pipeline Execution

In [ ]:
pipeline = PreprocessingPipeline(mode='photo', binarization_method='sauvola')
result = pipeline.process(img_photo)
plot_preprocessing_stages(result, figsize=(16, 12))
print("Metadata Summary:", result.metadata)